# ResNet-50 CNN — CUB-200-2011 Bird Photos (R4)

This notebook fine-tunes an **ImageNet-pretrained ResNet-50** to classify the 200 bird species from their photos, and compares it with the attribute-based models (decision tree, random forest, KNN, logistic regression, MLP).

**Main metric:** macro-F1 on the official test split (accuracy and top-5 accuracy are reported alongside).

**Input:** the cropped photos made by the team's image pipeline (`imagePrep.ipynb` → `scripts/prepare_images.py`). The notebook runs both steps itself if their output is missing. Following *Decision 9* in [`05_image_preparation.ipynb`](05_image_preparation.ipynb), every photo (train **and** test) is cropped to its bounding box plus a 10% margin. The report should say so, because the boxes are extra information that a plain photo would not have.

### Two profiles

| | `test` (Colab T4) | `full` (Colab L4) |
|---|---|---|
| Purpose | quick check of the code and a first result | final model for the report |
| Input size | 224 px (photos stored at 256 px) | 448 px (photos stored at 512 px) |
| Batch size | 64 | 32 (448 px photos need 4× the memory) |
| Epochs | 15 | 25 |
| Backbone learning rates tried | 0.002 | 0.001 and 0.003 |
| Final model | best validation epoch, trained on 80% of the training photos | retrained on **all** training photos with the chosen learning rate |
| Result (official test split) | 76.4% accuracy, macro-F1 0.764 | — |

In both, the new 200-class layer learns 10× faster than the pretrained layers. Setting `PRETRAINED = False` (test profile only) trains the same network **from scratch** for comparison.

### Protocol (no test leakage)
1. **Select** (section 6): for each learning rate, train on 80% of the official training photos and score the other 20% (the validation set from `make_splits` in `scripts/augmentation.py`) after every epoch. The learning rate with the best validation macro-F1 wins.
2. **Final** (section 7, `full` only): retrain from the ImageNet weights on all 5,994 training photos with that learning rate, for the same number of epochs, and keep the last epoch. With a cosine schedule the last epochs barely change the model (the test run's best epoch was 14 of 15, 0.001 macro-F1 better than epoch 15), so no validation set is needed to choose an epoch.
3. **Test** (section 8): the official test photos are scored **once**, at the end.

### How to run on Colab
1. Open this notebook from GitHub in Colab (File → Open notebook → GitHub → `Flamingsides/f20dl-coursework-1`, branch `Ryan`).
2. Runtime → Change runtime type → **L4 GPU** for `full` (or **T4 GPU** for `test`).
3. The first time only: put your `kaggle.json` in Google Drive at `MyDrive/f20dl_resnet50/kaggle.json`.
4. Runtime → Run all. The first session for each profile downloads and crops the photos (~10 min) and caches them in Drive; later sessions restore the cache in about a minute.

If Colab disconnects, run all again: every run resumes from its last epoch saved in Drive, and finished runs are skipped.

## 1. Settings

In [ ]:
# --- GenAI disclosure (F20DL coursework spec, Section 8) ---
# This notebook was drafted with Claude (Anthropic) in a guided conversation, then reviewed,
# run and checked by Ryan. Main prompts used:
# 1. "I want to create a CNN model that uses the ResNet-50 network. Help me research how it can be
#     best implemented and create a plan for it."
# 2. "I need it to achieve the best accuracy possible based on the F1 score of the model"
# 3. "It can be a fine-tuned version of a pretrained ResNet-50 but give some options and also compare it
#     to an estimation of how it would compare to the accuracy of one trained from scratch."
# 4. "let's stick to your plan of using the T4 for testing and the L4 for the full run.
#     You may start with the code for the test run now"
# 5. "Set up full run"

PROFILE = "full"       # "test" = Colab T4, 224 px quick check | "full" = Colab L4, 448 px final model
PRETRAINED = True      # False = same network trained from scratch (comparison run, test profile only)
SMOKE_TEST = False     # True = tiny subset, 1 epoch: only checks that the code runs end to end

REPO_URL = "https://github.com/Flamingsides/f20dl-coursework-1.git"
BRANCH = "Ryan"
DRIVE_FOLDER = "f20dl_resnet50"   # folder in MyDrive for the data cache, checkpoints and results

PROFILES = {
    #        input px, stored px, batch, epochs, backbone learning rates to try, retrain on all training photos
    "test": dict(img_size=224, store_px=256, batch_size=64, epochs=15, lr_grid=[0.002], final_retrain=False),
    "full": dict(img_size=448, store_px=512, batch_size=32, epochs=25, lr_grid=[0.001, 0.003], final_retrain=True),
}
P = PROFILES[PROFILE]
IMG_SIZE, STORE_PX, BATCH_SIZE, EPOCHS = P["img_size"], P["store_px"], P["batch_size"], P["epochs"]
LR_GRID, FINAL_RETRAIN = P["lr_grid"], P["final_retrain"]
WARMUP_EPOCHS, WEIGHT_DECAY = 1, 1e-4
HEAD_LR_MULT = 10            # the new 200-way classifier starts from random, so it learns faster
LABEL_SMOOTHING = 0.1
WEIGHTS = "IMAGENET1K_V1"    # the standard ImageNet ResNet-50 weights used in most CUB papers

if not PRETRAINED:
    assert PROFILE == "test", "the from-scratch comparison is only set up for the test profile"
    EPOCHS, WARMUP_EPOCHS, WEIGHT_DECAY = 60, 3, 5e-4   # more regularisation: ~24 photos per species is very little
    LR_GRID, HEAD_LR_MULT = [0.1 * BATCH_SIZE / 256], 1  # standard ImageNet-from-scratch rate, scaled to the batch

if SMOKE_TEST:
    EPOCHS, WARMUP_EPOCHS, BATCH_SIZE = 1, 0, 8

BASE_NAME = f"resnet50_{'pretrained' if PRETRAINED else 'scratch'}_{IMG_SIZE}px" + ("_smoke" if SMOKE_TEST else "")
CONFIG = dict(profile=PROFILE, pretrained=PRETRAINED, weights=WEIGHTS if PRETRAINED else None, img_size=IMG_SIZE,
              store_px=STORE_PX, batch_size=BATCH_SIZE, epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS,
              head_lr_mult=HEAD_LR_MULT, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING,
              smoke_test=SMOKE_TEST)
print(f"{BASE_NAME}: backbone learning rates {LR_GRID} | final retrain on all training photos: {FINAL_RETRAIN}")
CONFIG

## 2. Environment

On Colab: mount Google Drive and clone the repository (for the team's image-pipeline code). Locally: use this repository.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE.mkdir(parents=True, exist_ok=True)
    ROOT = Path("/content/f20dl-coursework-1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    OUT = DRIVE                                    # survives disconnects
    CKPT_DIR = DRIVE / "checkpoints"
else:
    DRIVE = None
    ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
    OUT = ROOT / "results" / "resnet50"
    CKPT_DIR = ROOT / "data" / "processed" / "checkpoints"   # git-ignored: checkpoints are ~100-200 MB

FIGURES = OUT / "figures"
for d in (OUT, CKPT_DIR, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts"))

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True          # picks the fastest convolution algorithms for fixed-size inputs
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"                    # mixed precision: ~2x faster, same accuracy
GPU = torch.cuda.get_device_name() if DEVICE.type == "cuda" else ""

print(f"Colab: {IN_COLAB} | device: {DEVICE} {GPU} | {os.cpu_count()} CPU cores")
print(f"torch {torch.__version__} | outputs -> {OUT}")
if IN_COLAB and DEVICE.type != "cuda":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> L4 GPU (full) or T4 GPU (test), then run all again.")
if PROFILE == "full" and "T4" in GPU and not SMOKE_TEST:
    print("Warning: the full profile is set up for an L4. On a T4 it runs, but roughly 2-3x slower.")

## 3. Data

The cropped photos are made by the team's image pipeline: `imagePrep.ipynb` checks every photo and writes `data/cub_images.csv`, then `scripts/prepare_images.py --short-side <px>` converts to RGB, crops to the bounding box + 10% margin and resizes the shorter side (256 px for `test`, 512 px for `full`, so the 448 px random crops still have room). On Colab the result is cached in Drive as a zip, so this only runs once per profile.

Colab does not show the output of these helper programs while they run, so this cell can look idle for up to ~10 minutes the first time.

In [ ]:
SUFFIX = "" if STORE_PX == 256 else f"_{STORE_PX}"
PROCESSED = ROOT / "data" / "processed"
CSV = PROCESSED / f"cub_images_cropped{SUFFIX}.csv"
IMG_DIR = PROCESSED / f"images_cropped{SUFFIX}"
CACHE_ZIP = DRIVE / f"cub_processed_{STORE_PX}px.zip" if IN_COLAB else None

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, cwd=ROOT)

if not CSV.exists() and IN_COLAB and CACHE_ZIP.exists():
    print(f"Restoring {STORE_PX} px photos from the Drive cache...")
    shutil.unpack_archive(CACHE_ZIP, ROOT / "data")

if not CSV.exists():
    has_raw = any((ROOT / "data" / "raw").rglob("image_class_labels.txt")) or (ROOT / "CUB_200_2011").exists()
    if not has_raw:
        if IN_COLAB:
            token = DRIVE / "kaggle.json"
            if not token.exists():
                raise FileNotFoundError(f"Put your kaggle.json in Google Drive at {token} (see the top of this notebook)")
            kaggle_dir = Path.home() / ".kaggle"
            kaggle_dir.mkdir(exist_ok=True)
            shutil.copy(token, kaggle_dir / "kaggle.json")
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
        run([sys.executable, "scripts/download_data.py"])
    if not (ROOT / "data" / "cub_images.csv").exists():
        run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", PROCESSED / "_executed",
             "notebooks/imagePrep.ipynb"])
    run([sys.executable, "scripts/prepare_images.py", "--short-side", STORE_PX])
    if IN_COLAB:
        print("Caching the cropped photos in Drive...", flush=True)
        local_zip = Path("/content") / CACHE_ZIP.name     # build locally, then one copy to Drive (much faster)
        with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:   # JPEGs are already compressed
            z.write(CSV, CSV.relative_to(ROOT / "data"))
            for p in IMG_DIR.rglob("*"):
                if p.is_file():
                    z.write(p, p.relative_to(ROOT / "data"))
        shutil.copy(local_zip, CACHE_ZIP)

print(f"{CSV.relative_to(ROOT)}: {len(pd.read_csv(CSV)):,} photos")

## 4. Splits and augmentation

The splits and dataset class come from `scripts/augmentation.py`, so this model uses exactly the same train / validation / test photos as the rest of the image pipeline. The transforms repeat that file's choices (random crop, horizontal flip, ±15° rotation, lighting jitter with almost no hue change, random erasing) but take the image size as a parameter.

In [ ]:
from torchvision.transforms import v2 as T
from augmentation import BirdImages, make_splits, IMAGENET_MEAN, IMAGENET_STD, N_CLASSES, RANDOM_STATE

def make_transforms(size):
    """Same augmentation choices as scripts/augmentation.py, for any input size."""
    train = T.Compose([
        T.ToImage(),
        T.RandomResizedCrop(size, scale=(0.6, 1.0), ratio=(3 / 4, 4 / 3), antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])
    evaluate = T.Compose([                                     # validation and test: no randomness
        T.ToImage(),
        T.Resize(round(size * 256 / 224), antialias=True),     # 224 -> 256 (as in augmentation.py), 448 -> 512
        T.CenterCrop(size),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train, evaluate

train_df, val_df, test_df = make_splits(CSV)
if SMOKE_TEST:   # 10 species, a few photos each
    pick = lambda df, n: df[df.class_id <= 10].groupby("class_id").head(n)
    train_df, val_df, test_df = pick(train_df, 4), pick(val_df, 2), pick(test_df, 3)
full_train_df = pd.concat([train_df, val_df])     # all official training photos, for the final retrain
assert not set(full_train_df.image_id) & set(test_df.image_id)
assert not set(train_df.image_id) & set(val_df.image_id)

train_tf, eval_tf = make_transforms(IMG_SIZE)
NUM_WORKERS = 0 if SMOKE_TEST else min(12, os.cpu_count() or 2)

def make_loader(df, train):
    return DataLoader(BirdImages(df, PROCESSED, train_tf if train else eval_tf), batch_size=BATCH_SIZE,
                      shuffle=train, drop_last=train, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0,
                      generator=torch.Generator().manual_seed(RANDOM_STATE) if train else None)

species = (pd.read_csv(CSV)[["class_id", "species"]].drop_duplicates()
           .set_index("class_id").sort_index().species.tolist())       # index = class_id - 1
print(f"train {len(train_df):,} | val {len(val_df):,} | all training {len(full_train_df):,} | "
      f"test {len(test_df):,} photos | {NUM_WORKERS} loader workers")

## 5. Model and training code

- **ResNet-50** with its 1000-class ImageNet classifier replaced by a new 200-class layer.
- **Two learning rates:** the pretrained layers move 10× more slowly than the new classifier, so fine-tuning adjusts the ImageNet features instead of overwriting them.
- **SGD with momentum 0.9**, one warm-up epoch, then cosine decay to zero (updated every batch).
- **Label smoothing 0.1:** the target is 90% on the true species and the rest spread over the others, which discourages over-confident memorising of ~30 photos per species.
- **Mixed precision** and channels-last memory layout for speed on the GPU.
- Each run saves a checkpoint to Drive after every epoch and resumes from it after a disconnect.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def new_model():
    torch.manual_seed(RANDOM_STATE)
    model = resnet50(weights=ResNet50_Weights[WEIGHTS] if PRETRAINED else None, progress=False)
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    return model.to(DEVICE, memory_format=torch.channels_last)

def new_optimiser(model, lr_backbone, steps_per_epoch):
    head = list(model.fc.parameters())
    head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.SGD([{"params": backbone, "lr": lr_backbone},
                                 {"params": head, "lr": lr_backbone * HEAD_LR_MULT}],
                                momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
    steps, warmup = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        seen += len(y)
    return loss_sum / seen, correct / seen

@torch.no_grad()
def predict(model, loader):
    """Class probabilities, true labels (0-199) and mean cross-entropy (without smoothing)."""
    model.eval()
    probs, labels, loss_sum = [], [], 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
        loss_sum += nn.functional.cross_entropy(logits.float(), y, reduction="sum").item()
        probs.append(logits.float().softmax(1).cpu())
        labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return probs, labels, loss_sum / len(labels)

def scores(probs, labels):
    preds = probs.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "top5_accuracy": top_k_accuracy_score(labels, probs, k=5, labels=np.arange(N_CLASSES))}

def save_atomic(obj, path):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def fit(run_name, train_rows, val_rows, lr_backbone):
    """Train one run (resumable). With validation rows, keep the epoch with the best validation macro-F1;
    without, keep the last epoch. Returns the epoch history and a summary; the kept weights are in <run>_best.pt."""
    config = {**CONFIG, "run": run_name, "lr_backbone": lr_backbone, "train_photos": len(train_rows)}
    last, best_path = CKPT_DIR / f"{run_name}_last.pt", CKPT_DIR / f"{run_name}_best.pt"
    train_loader = make_loader(train_rows, train=True)
    val_loader = make_loader(val_rows, train=False) if val_rows is not None else None
    model = new_model()
    optimizer, scheduler = new_optimiser(model, lr_backbone, len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start, history, best = 0, [], {"epoch": 0, "val": None}
    if last.exists():
        ckpt = torch.load(last, map_location=DEVICE, weights_only=False)
        if ckpt["config"] != config:
            raise ValueError(f"{last.name} was saved with different settings - delete it to start this run again")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start, history, best = ckpt["epoch"] + 1, ckpt["history"], ckpt["best"]
        print(f"{run_name}: resuming after epoch {start}" if start < EPOCHS else f"{run_name}: already finished")

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        row = {"epoch": epoch + 1, "train_loss": train_loss, "train_accuracy": train_acc}
        if val_loader is not None:
            val_probs, val_labels, val_loss = predict(model, val_loader)
            val = scores(val_probs, val_labels)
            row.update(val_loss=val_loss, **{f"val_{k}": v for k, v in val.items()})
        row["seconds"] = time.time() - t0
        history.append(row)
        keep = val_loader is None or best["val"] is None or val["macro_f1"] > best["val"]["macro_f1"]
        if keep:
            best = {"epoch": epoch + 1, "val": val if val_loader is not None else None}
            save_atomic({"model": model.state_dict(), "config": config, **best}, best_path)
        save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                     "history": history, "best": best, "config": config}, last)
        msg = f"{run_name} | epoch {epoch + 1:>2}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f}"
        if val_loader is not None:
            msg += (f" | val loss {val_loss:.3f} acc {val['accuracy']:.3f} macro-F1 {val['macro_f1']:.3f}"
                    f" top-5 {val['top5_accuracy']:.3f}")
        print(msg + f" | {row['seconds']:.0f}s" + ("  *" if keep and val_loader is not None else ""), flush=True)

    history = pd.DataFrame(history)
    history.to_csv(OUT / f"{run_name}_history.csv", index=False)
    del model, optimizer, scheduler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return history, {"run": run_name, "lr_backbone": lr_backbone, "best_epoch": best["epoch"], "val": best["val"],
                     "train_minutes": round(history.seconds.sum() / 60, 1)}

## 6. Select the learning rate (train on 80%, score on validation)

For the `full` profile this trains two runs: the test run's learning rate (scaled to the smaller batch) and one 3× higher, because the test run levelled off early, which can mean the backbone was adapting too slowly.

In [ ]:
selection = {}
for lr in LR_GRID:
    name = BASE_NAME if len(LR_GRID) == 1 else f"{BASE_NAME}_lr{lr:g}"
    selection[lr] = fit(name, train_df, val_df, lr)

selection_table = pd.DataFrame([{"backbone lr": lr, "run": s["run"], "best epoch": s["best_epoch"],
                                 "val accuracy": s["val"]["accuracy"], "val macro-F1": s["val"]["macro_f1"],
                                 "val top-5": s["val"]["top5_accuracy"], "minutes": s["train_minutes"]}
                                for lr, (_, s) in selection.items()])
BEST_LR = max(selection, key=lambda lr: selection[lr][1]["val"]["macro_f1"])
print(f"Chosen backbone learning rate: {BEST_LR:g}")
selection_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for lr, (hist, s) in selection.items():
    line, = axes[0].plot(hist.epoch, hist.val_loss, label=f"validation, lr {lr:g}")
    axes[0].plot(hist.epoch, hist.train_loss, ":", color=line.get_color(), label=f"train (smoothed labels), lr {lr:g}")
    line, = axes[1].plot(hist.epoch, hist.val_macro_f1, label=f"validation macro-F1, lr {lr:g}")
    axes[1].plot(hist.epoch, hist.train_accuracy, ":", color=line.get_color(), label=f"train accuracy, lr {lr:g}")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation macro-F1 and training accuracy", xlabel="epoch", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
fig.suptitle(f"{BASE_NAME}: learning-rate selection", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES / f"{BASE_NAME}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

A widening gap between training accuracy and validation macro-F1 means the network is memorising the training photos (overfitting). Validation macro-F1 levelling off shows where more epochs stop helping.

## 7. Final model (`full`: retrain on all training photos)

The validation photos are 20% of the training data. Once the learning rate is chosen they are not needed any more, so the final model is trained again from the ImageNet weights on all 5,994 training photos (~25% more photos per species). The `test` profile keeps the selected run's best-validation-epoch weights instead.

In [ ]:
if FINAL_RETRAIN:
    FINAL_NAME = f"{BASE_NAME}_final"
    final_history, final_summary = fit(FINAL_NAME, full_train_df, None, BEST_LR)
else:
    FINAL_NAME = selection[BEST_LR][1]["run"]
    final_history, final_summary = selection[BEST_LR]
print(f"Final model: {FINAL_NAME} ({final_summary['train_minutes']} min)")

## 8. Final test evaluation (run once)

The final model is scored once on the 5,794 official test photos.

In [ ]:
model = new_model()
final_ckpt = torch.load(CKPT_DIR / f"{FINAL_NAME}_best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(final_ckpt["model"])
test_probs, test_labels, test_loss = predict(model, make_loader(test_df, train=False))
test = scores(test_probs, test_labels)

results = {"run": FINAL_NAME, "config": {**CONFIG, "lr_backbone": BEST_LR, "final_retrain": FINAL_RETRAIN},
           "selection": selection_table.to_dict("records"), "final_epoch": final_ckpt["epoch"],
           "test": test, "test_photos": int(len(test_labels)),
           "train_minutes_total": round(sum(s["train_minutes"] for _, s in selection.values())
                                        + (final_summary["train_minutes"] if FINAL_RETRAIN else 0), 1)}
(OUT / f"{FINAL_NAME}_results.json").write_text(json.dumps(results, indent=2))

per_class = pd.DataFrame({"class_id": np.arange(1, N_CLASSES + 1), "species": species,
                          "f1": f1_score(test_labels, test_probs.argmax(1), average=None, labels=np.arange(N_CLASSES))})
per_class.to_csv(OUT / f"{FINAL_NAME}_per_class_f1.csv", index=False)
print(f"{FINAL_NAME} | test accuracy {test['accuracy']:.4f} | macro-F1 {test['macro_f1']:.4f} | "
      f"top-5 {test['top5_accuracy']:.4f}")

# Best attribute-based results on the same official test split (from the notebooks named), plus every ResNet run
# whose results file is in the output folder (e.g. the test run)
rows = [("Random forest, tuned (randomForest.ipynb)", "312 attributes", 0.4563, 0.4349, 0.7206),
        ("RF + logistic regression ensemble (randomForestOptimisation.ipynb)", "attributes + aggregates",
         0.5104, 0.5059, 0.7656)]
for path in sorted(OUT.glob("resnet50_*_results.json")):
    r = json.loads(path.read_text())
    rows.append((r["run"], "photos", r["test"]["accuracy"], r["test"]["macro_f1"], r["test"]["top5_accuracy"]))
pd.DataFrame(rows, columns=["model", "input", "test accuracy", "test macro-F1", "test top-5"]).round(4)

## 9. Which species does it confuse?

The most frequent test mistakes, as (true species → predicted species). On CUB these are usually sibling species from the same family, which matches what the attribute models found.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, test_probs.argmax(1), labels=np.arange(N_CLASSES))
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm, axis=None)[::-1][:10], cm.shape))[0]
print(pd.DataFrame([(species[t], species[p], cm[t, p], int((test_labels == t).sum())) for t, p in top],
                   columns=["true species", "predicted as", "mistakes", "test photos of true species"]).to_string())

## 10. Files written

All in `MyDrive/f20dl_resnet50/` on Colab (`results/resnet50/` locally). Copy the small ones into the repo's `results/resnet50/` for the report:
- `<run>_results.json`: settings, learning-rate selection, test scores
- `<run>_history.csv`: one row per epoch for every run
- `<run>_per_class_f1.csv`: test F1 for each species
- `figures/<base>_curves.png`: learning curves

The `checkpoints/` folder and the `cub_processed_*.zip` caches are large and only needed on Colab.